In [14]:
import torch
import torch.nn as nn
from torchvision import models

## Load ResNet model

In [15]:
model = models.resnet18(weights="DEFAULT")
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [16]:
model.fc = nn.Linear(
    model.fc.in_features,
    2
)
print(model.fc)

Linear(in_features=512, out_features=2, bias=True)


## Image transformation
### from [1 * 28 * 28] to [3 * 224 * 224]

In [17]:
from torchvision import transforms

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Upsampling
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

## ResNet DataLoaders

In [18]:
from medmnist.dataset import PneumoniaMNIST
from torch.utils.data import DataLoader

train_dataset_resnet = PneumoniaMNIST(
    split="train",
    transform=resnet_transform,
    download=True
)

val_dataset_resnet = PneumoniaMNIST(
    split="val",
    transform=resnet_transform,
    download=True
)

test_dataset_resnet = PneumoniaMNIST(
    split="test",
    transform=resnet_transform,
    download=True
)

In [19]:
train_loader_resnet = DataLoader(
    train_dataset_resnet,
    batch_size=32,
    shuffle=True
)

val_loader_resnet = DataLoader(
    val_dataset_resnet,
    batch_size=32,
    shuffle=False
)

test_loader_resnet = DataLoader(
    test_dataset_resnet,
    batch_size=32,
    shuffle=False
)

In [20]:
images, labels = next(iter(train_loader_resnet))

print("Image shape:", images.shape)
print("Label shape:", labels.shape)

Image shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32, 1])


## Transfer Learning

In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [22]:
from tqdm.auto import tqdm

In [23]:
# Train ResNet-18

num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0

    progress_bar = tqdm(
        train_loader_resnet,
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for imgs, labels in train_loader_resnet:
        labels = labels.long().view(-1)

        outputs = model(imgs)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / len(train_loader_resnet)
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {epoch_loss:.4f}"
    )

Epoch 1/10:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch [1/10] Train Loss: 0.1427


Epoch 2/10:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch [2/10] Train Loss: 0.0748


Epoch 3/10:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch [3/10] Train Loss: 0.0594


... (10 outputs left out)

Epoch 9/10:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch [9/10] Train Loss: 0.0419


Epoch 10/10:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch [10/10] Train Loss: 0.0077


## Validation

In [25]:
model.eval()

correct_labels = 0
total_labels = 0
running_val_loss = 0.0

with torch.no_grad():

    for images, labels in val_loader_resnet:

        labels = labels.long().view(-1)

        outputs = model(images)

        loss = criterion(outputs, labels)

        running_val_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)


val_loss = running_val_loss / len(val_loader_resnet)

val_accuracy = correct_labels / total_labels * 100

print(f"Val Loss: {val_loss:.4f}")
print(f"Val Accuracy: {val_accuracy:.2f}%")

Val Loss: 0.0687
Val Accuracy: 97.71%


## Test Evaluation

In [26]:
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():

    for images, labels in test_loader_resnet:

        labels = labels.long().view(-1)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)[:, 1]
        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())


# Confusion Matrix
cm = confusion_matrix(all_labels, all_predictions)

print("Confusion Matrix:")
print(cm)


# Accuracy
accuracy = accuracy_score(all_labels, all_predictions)

# TN, FP, FN, TP
TN, FP, FN, TP = cm.ravel()

# Sensitivity
sensitivity = TP / (TP + FN)

# Specificity
specificity = TN / (TN + FP)

# ROC-AUC
roc_auc = roc_auc_score(all_labels, all_probabilities)

print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Sensitivity: {sensitivity * 100:.2f}%")
print(f"Specificity: {specificity * 100:.2f}%")
print(f"ROC-AUC: {roc_auc:.4f}")

Confusion Matrix:
[[138  96]
 [  0 390]]
Accuracy: 84.62%
Sensitivity: 100.00%
Specificity: 58.97%
ROC-AUC: 0.9639


In [ ]:
torch.save(
    model.state_dict(),
    "../results/resnet18_baseline.pth"
)

print("resnet18 CNN saved successfully.")